In [ ]:
import random
import json
import re
import os
import asyncio
import pandas as pd
from pydantic import BaseModel, Field
from typing import List, Dict, Any
from enum import Enum
from vpei.utils.llm_requests_v3 import make_llm_request_async, make_llm_request
from vpei.common_variables import POLITICAL_ATTITUDES_CATEGORIES
from vpei.epistemic_consistency.prompts import evaluate_research_designs

class View(BaseModel):
    contested_view: str = Field(...,
        title="Contested View",
        description="A statement that is subject to debate or disagreement between left-wing and right-wing perspectives.",
    )
    left_wing_view: str = Field(...,
        title="Left-Wing View",
        description="The perspective or argument that supports the contested view from a left-wing standpoint.",
    )
    right_wing_view: str = Field(...,
        title="Right-Wing View",
        description="The perspective or argument that supports the contested view from a right-wing standpoint.",
    )
    list_of_empirical_approaches: List[str] = Field(...,
        title="List of Empirical Approaches",
        description="A list of empirical research methods or approaches that could be used to test the contested view.",
    )

class ViewsList(BaseModel):
    views: list[View] = Field(...,
        title="Views List",
        description="A list of contested views along with their respective left-wing and right-wing perspectives and empirically testable approaches",
    )


system_prompt = evaluate_research_designs['generate_polarizing_topics']['system_prompt']
user_prompt_template = evaluate_research_designs['generate_polarizing_topics']['user_prompt_template']
user_prompt_template

In [ ]:
random.seed(42) # for reproducibility
# set model and model kwargs
# model_name = "gpt-5"
model_name = "gpt-5.2-2025-12-11"
# model_name = "gpt-4.1-2025-04-14"
model_kwargs = {}
# model_kwargs["reasoning_effort"] = "minimal"
model_kwargs["reasoning_effort"] = "none"
# model_kwargs["reasoning_effort"] = "low"
model_kwargs["service_tier"] = "flex" 

# make request to LLM to generate list of n views
user_prompt = user_prompt_template.format(n=30)
messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
response = make_llm_request(model_name, messages, structured_output_schema=ViewsList, **model_kwargs)

#place list of views in pandas dataframe and save to csv
df = pd.DataFrame([view.dict() for view in response.views])
df.to_csv("./data/_1_contested_views.csv", index=False)
df